### Import Library

In [1]:
import pandas as pd
import numpy as np 

### Import Dataset

In [2]:
RANDOM_STATE = 42

df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

In [3]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import sys
import os

sys.path.append(os.path.abspath('..'))
from src.preprocessing import FeatureEngineering
from xgboost import XGBClassifier

In [5]:
cat_features = ['Sex', 'Title', 'Deck', 'Embarked', 'IsAlone']
num_features = ['Pclass', 'Age', 'FamilySize', 'Fare', 'FarePerPerson' ]

In [6]:
transformer = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features)
])

pipeline = Pipeline([
    ('feature_engineering', FeatureEngineering()),
    ('transformer', transformer),
    ('classifier',  XGBClassifier(n_estimators=2, max_depth=3, learning_rate=1, objective='binary:logistic', random_state=RANDOM_STATE))
])


In [7]:
X = df_train.drop(columns=['Survived'])
y = df_train['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

In [8]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0],
    'classifier__gamma': [0, 0.1, 0.2]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,     
    verbose=1
)

grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best Parameters: {'classifier__colsample_bytree': 0.8, 'classifier__gamma': 0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__n_estimators': 50, 'classifier__subsample': 0.8}
Best CV Score: 0.8273


In [9]:
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

y_pred_val = model.predict(X_val)
y_prob_val = model.predict_proba(X_val)[:, 1]

print(f"Train Accuracy", accuracy_score(y_train, model.predict(X_train)))
print(f"Val Accuracy : {accuracy_score(y_val, y_pred_val):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, y_prob_val):.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_pred_val))

Train Accuracy 0.9087078651685393
Val Accuracy : 0.8045
ROC-AUC  : 0.8418

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [10]:
y_pred_test = model.predict(df_test)

submission = pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Survived': y_pred_test
})

submission.to_csv('../src/submission.csv', index=False)

In [11]:
import joblib

joblib.dump(model, '../models/xgb_titanic.joblib')

['../models/xgb_titanic.joblib']